# 🧹 Delhivery Logistics Network — Data Cleaning
### Notebook 2 

**Objective:** Transform raw, messy data into a clean, analysis-ready dataset.  
Every cleaning decision is documented with a clear reason.  
No silent drops, no unexplained transformations.

**Key issues identified in exploration:**
- Extreme outliers in delay ratios (segment max: 574x, full trip max: 77x)
- Potential duplicate rows
- Timestamp columns stored as strings
- Missing values in certain columns
- Inconsistent facility naming formats

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

os.makedirs('../outputs/visualisations', exist_ok=True)
os.makedirs('../outputs/model_results', exist_ok=True)

print("✅ Libraries loaded")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

## 📂 Step 1 — Load Raw Data

We load the original raw file and immediately make a copy.  
All cleaning is done on the copy — raw data is never modified.  
This is standard practice in any professional DS workflow.

In [ ]:
# Load raw dataset
raw = pd.read_csv('../data/delivery_data.csv')

# Make a working copy — never touch raw
df = raw.copy()

print("=" * 55)
print("         RAW DATASET LOADED")
print("=" * 55)
print(f"\n  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print(f"\n  Shape before cleaning: {df.shape}")
print("\n" + "=" * 55)

## 🔍 Step 2 — Baseline Quality Audit

Before cleaning anything, we document the exact state  
of the data. This becomes the "before" in our before/after comparison.

In [ ]:
# Baseline audit
print("BASELINE DATA QUALITY AUDIT")
print("=" * 55)

# Missing values
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
print("\n1. MISSING VALUES:")
print("-" * 40)
for col in df.columns:
    if missing[col] > 0:
        print(f"   {col:<40} {missing[col]:>6,} ({missing_pct[col]:.2f}%)")
if missing.sum() == 0:
    print("   No missing values found")

# Duplicates
dupes = df.duplicated().sum()
print(f"\n2. DUPLICATE ROWS: {dupes:,}")

# Data types
print("\n3. DATA TYPES:")
print("-" * 40)
for col, dtype in df.dtypes.items():
    print(f"   {col:<40} {str(dtype)}")

print("\n" + "=" * 55)

## 🗑️ Step 3 — Remove Duplicates

Duplicate rows in logistics data usually mean  
the same trip segment was recorded twice — a data pipeline issue.  
We identify and remove them, documenting exactly how many.

In [ ]:
before = len(df)
dupes  = df.duplicated().sum()

print(f"Duplicate rows found: {dupes:,}")

if dupes > 0:
    df = df.drop_duplicates()
    print(f"✅ Removed {dupes:,} duplicate rows")
else:
    print("✅ No duplicates found — no action needed")

after = len(df)
print(f"\n  Rows before : {before:,}")
print(f"  Rows after  : {after:,}")
print(f"  Rows removed: {before - after:,}")

## ⏰ Step 4 — Fix Timestamp Columns

All time columns are currently stored as strings.  
We convert them to proper datetime objects.  
This is required for any time-based feature engineering later.

In [ ]:
time_cols = [
    'trip_creation_time',
    'od_start_time',
    'od_end_time',
    'cutoff_timestamp'
]

print("Converting timestamp columns:\n")
for col in time_cols:
    if col in df.columns:
        before_dtype = df[col].dtype
        df[col] = pd.to_datetime(df[col], errors='coerce')
        after_dtype = df[col].dtype
        null_after = df[col].isnull().sum()
        print(f"  ✅ {col}")
        print(f"     {str(before_dtype)} → {str(after_dtype)}")
        if null_after > 0:
            print(f"     ⚠️  {null_after} values couldn't be parsed → set to NaT")
    else:
        print(f"  ❌ {col} not found in dataframe")

print(f"\nDate range of data:")
print(f"  Start : {df['od_start_time'].min()}")
print(f"  End   : {df['od_start_time'].max()}")
print(f"  Span  : {(df['od_start_time'].max() - df['od_start_time'].min()).days} days")

## 🔢 Step 5 — Fix Numeric Columns

We verify all numeric columns are stored as proper float/int types.  
Any column that should be numeric but is stored as string gets converted.  
We also check for negative values in columns where negatives are impossible.

In [ ]:
numeric_cols = [
    'actual_distance_to_destination',
    'actual_time',
    'osrm_time',
    'osrm_distance',
    'factor',
    'segment_actual_time',
    'segment_osrm_time',
    'segment_osrm_distance',
    'segment_factor',
    'start_scan_to_end_scan',
    'cutoff_factor'
]

print("Numeric Column Validation:\n")
print("-" * 65)
print(f"  {'Column':<35} {'Type':<10} {'Min':>8} {'Max':>10} {'Negatives':>10}")
print("-" * 65)

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        col_min  = df[col].min()
        col_max  = df[col].max()
        neg_count = (df[col] < 0).sum()
        flag = " ⚠️" if neg_count > 0 else ""
        print(f"  {col:<35} {str(df[col].dtype):<10} {col_min:>8.2f} {col_max:>10.2f} {neg_count:>10}{flag}")

print("-" * 65)

## ⚠️ Step 6 — Handle Negative Values

Negative time or distance values are physically impossible  
in a logistics context. They indicate data recording errors.  
**Decision:** Set negative time and distance values to NaN  
so they don't corrupt our delay ratio calculations.

In [ ]:
# Columns where negative values are impossible
non_negative_cols = [
    'actual_time', 'osrm_time',
    'segment_actual_time', 'segment_osrm_time',
    'actual_distance_to_destination', 'osrm_distance',
    'segment_osrm_distance', 'start_scan_to_end_scan'
]

print("Handling negative values:\n")
total_fixed = 0

for col in non_negative_cols:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            df.loc[df[col] < 0, col] = np.nan
            total_fixed += neg_count
            print(f"  ✅ {col:<40} {neg_count:>5} negatives → NaN")
        else:
            print(f"  ✓  {col:<40} no negatives found")

print(f"\n  Total values set to NaN: {total_fixed:,}")
print(f"\n  Reasoning: Negative time/distance is physically impossible.")
print(f"  These values indicate sensor or recording errors.")
print(f"  Setting to NaN prevents them from corrupting delay ratios.")

## 📊 Step 7 — Recompute Delay Ratios on Clean Data

Now that negatives are handled, we recompute our core metrics.  
The delay ratios from exploration were computed on dirty data.  
These are the clean, trustworthy versions we'll use in all future notebooks.

In [ ]:
# Recompute delay ratios on clean data
df['delay_ratio'] = df['actual_time'] / df['osrm_time']
df['segment_delay_ratio'] = df['segment_actual_time'] / df['segment_osrm_time']

# Remove infinite values (division by zero)
df['delay_ratio'] = df['delay_ratio'].replace([np.inf, -np.inf], np.nan)
df['segment_delay_ratio'] = df['segment_delay_ratio'].replace([np.inf, -np.inf], np.nan)

# Remove zero or negative ratios
df.loc[df['delay_ratio'] <= 0, 'delay_ratio'] = np.nan
df.loc[df['segment_delay_ratio'] <= 0, 'segment_delay_ratio'] = np.nan

print("Clean Delay Ratio Statistics:\n")
print("-" * 50)
print(f"  {'Metric':<35} {'Value':>10}")
print("-" * 50)
print(f"  {'Full trip mean delay ratio':<35} {df['delay_ratio'].mean():>10.4f}")
print(f"  {'Full trip median delay ratio':<35} {df['delay_ratio'].median():>10.4f}")
print(f"  {'Full trip max delay ratio':<35} {df['delay_ratio'].max():>10.4f}")
print(f"  {'Segment mean delay ratio':<35} {df['segment_delay_ratio'].mean():>10.4f}")
print(f"  {'Segment median delay ratio':<35} {df['segment_delay_ratio'].median():>10.4f}")
print(f"  {'Segment max delay ratio':<35} {df['segment_delay_ratio'].max():>10.4f}")
print("-" * 50)

## 🎯 Step 8 — Outlier Treatment for Delay Ratios

From exploration we know:
- Full trip delay ratio max: 77.39 (before cleaning)
- Segment delay ratio max: 574.25 (before cleaning)

**Strategy:** Use the **99th percentile** as the cap.  
Values above the 99th percentile are Winsorized (capped, not dropped).  
We keep the rows but limit extreme values — this preserves sample size  
while preventing outliers from distorting graph edge weights.

**Why not drop them?**  
Dropping would remove real corridors from the graph.  
Capping keeps the corridor in the network but with a realistic delay value.

In [ ]:
# Compute percentile caps
p99_full    = df['delay_ratio'].quantile(0.99)
p99_segment = df['segment_delay_ratio'].quantile(0.99)
p01_full    = df['delay_ratio'].quantile(0.01)
p01_segment = df['segment_delay_ratio'].quantile(0.01)

print("Outlier Caps (1st and 99th Percentile):\n")
print(f"  Full trip   — lower cap  : {p01_full:.4f}")
print(f"  Full trip   — upper cap  : {p99_full:.4f}")
print(f"  Segment     — lower cap  : {p01_segment:.4f}")
print(f"  Segment     — upper cap  : {p99_segment:.4f}")

# Count values outside caps
print(f"\n  Full trip values above 99th pct   : {(df['delay_ratio'] > p99_full).sum():,}")
print(f"  Full trip values below 1st pct    : {(df['delay_ratio'] < p01_full).sum():,}")
print(f"  Segment values above 99th pct     : {(df['segment_delay_ratio'] > p99_segment).sum():,}")
print(f"  Segment values below 1st pct      : {(df['segment_delay_ratio'] < p01_segment).sum():,}")

In [ ]:
# Apply Winsorization
df['delay_ratio_clean'] = df['delay_ratio'].clip(
    lower=p01_full, upper=p99_full
)
df['segment_delay_ratio_clean'] = df['segment_delay_ratio'].clip(
    lower=p01_segment, upper=p99_segment
)

print("✅ Winsorization applied\n")
print("Before vs After Winsorization:\n")
print("-" * 55)
print(f"  {'Metric':<30} {'Before':>10} {'After':>10}")
print("-" * 55)
print(f"  {'Full trip mean':<30} {df['delay_ratio'].mean():>10.4f} {df['delay_ratio_clean'].mean():>10.4f}")
print(f"  {'Full trip max':<30} {df['delay_ratio'].max():>10.4f} {df['delay_ratio_clean'].max():>10.4f}")
print(f"  {'Full trip std':<30} {df['delay_ratio'].std():>10.4f} {df['delay_ratio_clean'].std():>10.4f}")
print(f"  {'Segment mean':<30} {df['segment_delay_ratio'].mean():>10.4f} {df['segment_delay_ratio_clean'].mean():>10.4f}")
print(f"  {'Segment max':<30} {df['segment_delay_ratio'].max():>10.4f} {df['segment_delay_ratio_clean'].max():>10.4f}")
print("-" * 55)

In [ ]:
# Before vs after distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full trip delay ratio
axes[0].hist(df['delay_ratio'].dropna(), bins=50,
             alpha=0.5, color='#E05C5C', label='Before (raw)', edgecolor='white')
axes[0].hist(df['delay_ratio_clean'].dropna(), bins=50,
             alpha=0.7, color='#4A90D9', label='After (winsorized)', edgecolor='white')
axes[0].axvline(1.0, color='green', linewidth=2, linestyle='--', label='Perfect (1.0)')
axes[0].set_title('Full Trip Delay Ratio\nBefore vs After Winsorization',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Delay Ratio', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Segment delay ratio
axes[1].hist(df['segment_delay_ratio'].dropna(), bins=50,
             alpha=0.5, color='#E05C5C', label='Before (raw)', edgecolor='white')
axes[1].hist(df['segment_delay_ratio_clean'].dropna(), bins=50,
             alpha=0.7, color='#4A90D9', label='After (winsorized)', edgecolor='white')
axes[1].axvline(1.0, color='green', linewidth=2, linestyle='--', label='Perfect (1.0)')
axes[1].set_title('Segment Delay Ratio\nBefore vs After Winsorization',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Delay Ratio', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Outlier Treatment — Before vs After', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/winsorization_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 🏭 Step 9 — Standardize Facility Names

Hub names have inconsistent formatting.  
We standardize them to ensure the same facility  
isn't treated as two different nodes in the graph.  
This is critical — one naming inconsistency can split  
a hub into two nodes, breaking the graph structure.

In [ ]:
def standardize_hub_name(name):
    if pd.isna(name):
        return name
    # Strip whitespace
    name = str(name).strip()
    # Uppercase
    name = name.upper()
    # Replace multiple spaces with single space
    name = ' '.join(name.split())
    return name

# Apply to hub columns
hub_cols = ['source_center', 'destination_center']

print("Standardizing hub names:\n")
for col in hub_cols:
    before_unique = df[col].nunique()
    df[col] = df[col].apply(standardize_hub_name)
    after_unique = df[col].nunique()
    print(f"  {col}")
    print(f"    Unique before: {before_unique:,}")
    print(f"    Unique after : {after_unique:,}")
    if before_unique != after_unique:
        print(f"    ✅ Merged {before_unique - after_unique} duplicate hub names")
    else:
        print(f"    ✓  No merges needed")
    print()

## 🗺️ Step 10 — Extract Geographic Features

We extract state information from hub name strings.  
This creates a usable geographic dimension for analysis.

In [ ]:
# Extract state from source_name and destination_name
df['source_state'] = df['source_name'].str.extract(r'\(([^)]+)\)')
df['dest_state']   = df['destination_name'].str.extract(r'\(([^)]+)\)')

# Clean state names
df['source_state'] = df['source_state'].str.strip().str.title()
df['dest_state']   = df['dest_state'].str.strip().str.title()

# Same state flag — useful feature for modeling
df['is_same_state'] = (df['source_state'] == df['dest_state']).astype(int)

print("Geographic Features Created:\n")
print(f"  source_state unique values : {df['source_state'].nunique()}")
print(f"  dest_state unique values   : {df['dest_state'].nunique()}")
print(f"  Same state trips           : {df['is_same_state'].sum():,} ({df['is_same_state'].mean()*100:.1f}%)")
print(f"  Cross state trips          : {(df['is_same_state']==0).sum():,} ({(df['is_same_state']==0).mean()*100:.1f}%)")
print(f"\nTop 5 source states:")
print(df['source_state'].value_counts().head().to_string())

## ⏰ Step 11 — Extract Time Features

We create time-based features from the cleaned timestamps.  
These will be used as edge attributes in the graph  
and as features in the ML model.

In [ ]:
# Time features from od_start_time
df['hour_of_day'] = df['od_start_time'].dt.hour
df['day_of_week'] = df['od_start_time'].dt.dayofweek  # 0=Monday, 6=Sunday
df['month']       = df['od_start_time'].dt.month
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# Time of day buckets
def time_bucket(hour):
    if pd.isna(hour):
        return 'Unknown'
    hour = int(hour)
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['hour_of_day'].apply(time_bucket)

# Trip duration from timestamps
df['trip_duration_mins'] = (
    df['od_end_time'] - df['od_start_time']
).dt.total_seconds() / 60

print("Time Features Created:\n")
print(f"  hour_of_day        : 0-23 (int)")
print(f"  day_of_week        : 0-6 (0=Monday)")
print(f"  month              : 1-12")
print(f"  is_weekend         : {df['is_weekend'].sum():,} weekend trips ({df['is_weekend'].mean()*100:.1f}%)")
print(f"  time_of_day        : {df['time_of_day'].value_counts().to_dict()}")
print(f"  trip_duration_mins : mean={df['trip_duration_mins'].mean():.1f} mins")

## 🔗 Step 12 — Create Corridor Key

We create a unique key for each source→destination pair.  
This becomes the primary identifier for graph edges  
and makes corridor-level aggregations much simpler.

In [ ]:
# Create corridor key
df['corridor_key'] = df['source_center'] + '→' + df['destination_center']

# Corridor statistics
corridor_count = df['corridor_key'].nunique()
trips_per_corridor = df.groupby('corridor_key').size()

print(f"Corridor Key Created:\n")
print(f"  Format          : source_center→destination_center")
print(f"  Unique corridors: {corridor_count:,}")
print(f"  Trips per corridor:")
print(f"    Min    : {trips_per_corridor.min()}")
print(f"    Mean   : {trips_per_corridor.mean():.1f}")
print(f"    Median : {trips_per_corridor.median():.1f}")
print(f"    Max    : {trips_per_corridor.max()}")
print(f"\nExample corridor keys:")
for key in df['corridor_key'].value_counts().head(5).index:
    print(f"  {key}")

## 📋 Step 13 — Handle Remaining Missing Values

After all transformations, we check what missing values remain  
and make a final documented decision on each.

In [ ]:
# Final missing value check
missing_final = df.isnull().sum()
missing_final_pct = missing_final / len(df) * 100

print("Remaining Missing Values After Cleaning:\n")
print("-" * 60)
print(f"  {'Column':<40} {'Count':>8} {'Pct':>8}")
print("-" * 60)

has_missing = False
for col in df.columns:
    if missing_final[col] > 0:
        has_missing = True
        print(f"  {col:<40} {missing_final[col]:>8,} {missing_final_pct[col]:>7.2f}%")

if not has_missing:
    print("  ✅ No missing values remain")

print("-" * 60)
print(f"\nTotal missing values: {missing_final.sum():,}")

In [ ]:
# Drop rows where core columns are null
# These are rows we cannot use for graph construction or modeling
core_cols = ['source_center', 'destination_center',
             'actual_time', 'osrm_time', 'route_type']

before_drop = len(df)
df = df.dropna(subset=core_cols)
after_drop  = len(df)

print(f"Dropping rows with null values in core columns:\n")
print(f"  Core columns  : {core_cols}")
print(f"  Rows before   : {before_drop:,}")
print(f"  Rows dropped  : {before_drop - after_drop:,}")
print(f"  Rows remaining: {after_drop:,}")
print(f"\n  Reasoning: Rows without source/destination/time data")
print(f"  cannot contribute to graph construction or ETA prediction.")
print(f"  Dropping them is the correct decision.")

## ✅ Step 14 — Final Clean Dataset Overview

In [ ]:
print("=" * 60)
print("         FINAL CLEAN DATASET OVERVIEW")
print("=" * 60)
print(f"""
SHAPE
  Rows    : {df.shape[0]:,}
  Columns : {df.shape[1]}

COLUMNS ADDED DURING CLEANING
  delay_ratio_clean         : winsorized full trip delay
  segment_delay_ratio_clean : winsorized segment delay
  source_state              : state of source hub
  dest_state                : state of destination hub
  is_same_state             : 1 if intrastate trip
  hour_of_day               : hour trip started
  day_of_week               : day (0=Monday)
  month                     : month number
  is_weekend                : 1 if weekend trip
  time_of_day               : Morning/Afternoon/Evening/Night
  trip_duration_mins        : actual duration in minutes
  corridor_key              : source→destination identifier

DATA QUALITY
  Duplicates removed        : documented above
  Negatives handled         : set to NaN
  Outliers treated          : winsorized at 1st/99th pct
  Timestamps converted      : to datetime
  Hub names standardized    : uppercase + stripped
  Missing core rows dropped : documented above
""")
print("=" * 60)
print("  NEXT STEP → 03_graph_construction.ipynb")
print("=" * 60)

In [ ]:
# Save clean dataset
clean_path = '../data/delivery_data_clean.csv'
df.to_csv(clean_path, index=False)

print(f"✅ Clean dataset saved to: {clean_path}")
print(f"   Shape: {df.shape}")
print(f"\n  This file will be loaded by all subsequent notebooks.")
print(f"  Never modify the raw file — always load from clean.")

In [ ]:
print("=" * 60)
print("         FINAL DATA QUALITY CHECK")
print("=" * 60)

# 1. Shape
print(f"\n1. SHAPE")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

# 2. Missing values
print(f"\n2. MISSING VALUES")
total_missing = df.isnull().sum().sum()
if total_missing == 0:
    print(f"   ✅ Zero missing values")
else:
    print(f"   ⚠️  {total_missing:,} missing values remain")
    missing_cols = df.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0]
    for col, count in missing_cols.items():
        print(f"      {col:<40} {count:,}")

# 3. Duplicates
print(f"\n3. DUPLICATES")
dupes = df.duplicated().sum()
if dupes == 0:
    print(f"   ✅ Zero duplicate rows")
else:
    print(f"   ⚠️  {dupes:,} duplicates remain")

# 4. Negative values in key columns
print(f"\n4. NEGATIVE VALUES IN KEY COLUMNS")
check_cols = ['actual_time', 'osrm_time', 
              'segment_actual_time', 'segment_osrm_time',
              'actual_distance_to_destination']
all_clean = True
for col in check_cols:
    if col in df.columns:
        neg = (df[col] < 0).sum()
        if neg > 0:
            print(f"   ⚠️  {col:<40} {neg:,} negatives")
            all_clean = False
if all_clean:
    print(f"   ✅ No negative values in time/distance columns")

# 5. Delay ratio range
print(f"\n5. DELAY RATIO RANGE")
print(f"   delay_ratio_clean min  : {df['delay_ratio_clean'].min():.4f}")
print(f"   delay_ratio_clean max  : {df['delay_ratio_clean'].max():.4f}")
print(f"   delay_ratio_clean mean : {df['delay_ratio_clean'].mean():.4f}")
if df['delay_ratio_clean'].max() < 20:
    print(f"   ✅ No extreme outliers")
else:
    print(f"   ⚠️  Still has extreme values — check winsorization")

# 6. Data types
print(f"\n6. KEY COLUMN DATA TYPES")
key_cols = {
    'od_start_time'    : 'datetime64',
    'actual_time'      : 'float',
    'osrm_time'        : 'float',
    'route_type'       : 'object',
    'source_center'    : 'object',
    'destination_center': 'object'
}
for col, expected in key_cols.items():
    if col in df.columns:
        actual_type = str(df[col].dtype)
        ok = expected in actual_type
        icon = "✅" if ok else "⚠️ "
        print(f"   {icon} {col:<30} {actual_type}")

# 7. New columns check
print(f"\n7. ENGINEERED COLUMNS CHECK")
new_cols = [
    'delay_ratio_clean', 'segment_delay_ratio_clean',
    'source_state', 'dest_state', 'is_same_state',
    'hour_of_day', 'day_of_week', 'month',
    'is_weekend', 'time_of_day', 'corridor_key',
    'trip_duration_mins'
]
for col in new_cols:
    if col in df.columns:
        null_count = df[col].isnull().sum()
        icon = "✅" if null_count == 0 else "⚠️ "
        print(f"   {icon} {col:<35} nulls: {null_count:,}")
    else:
        print(f"   ❌ {col} — NOT FOUND")

print(f"\n{'=' * 60}")
if total_missing == 0 and dupes == 0 and all_clean:
    print("  🎉 DATASET IS CLEAN — READY FOR GRAPH CONSTRUCTION")
else:
    print("  ⚠️  SOME ISSUES REMAIN — REVIEW ABOVE")
print(f"{'=' * 60}")

In [ ]:
# Fix 1 — fill missing state with 'Unknown'
df['source_state'] = df['source_state'].fillna('Unknown')
df['dest_state']   = df['dest_state'].fillna('Unknown')

# Fix 2 — fill missing segment delay ratio with full trip ratio
# If segment ratio is missing, use the full trip ratio as proxy
df['segment_delay_ratio_clean'] = df['segment_delay_ratio_clean'].fillna(
    df['delay_ratio_clean']
)

print("✅ Remaining nulls handled")
print(f"   source_state nulls             : {df['source_state'].isnull().sum()}")
print(f"   dest_state nulls               : {df['dest_state'].isnull().sum()}")
print(f"   segment_delay_ratio_clean nulls: {df['segment_delay_ratio_clean'].isnull().sum()}")
print(f"\n   Total remaining nulls: {df.isnull().sum().sum()}")

In [ ]:
df.to_csv('../data/delivery_data_clean.csv', index=False)
print(f"✅ Clean dataset saved")
print(f"   Shape: {df.shape}")
print(f"   File : data/delivery_data_clean.csv")

---
## ✅ Data Cleaning Complete

### Summary of all decisions made:

| Issue | Decision | Reason |
|---|---|---|
| Duplicate rows | Dropped | Same segment recorded twice |
| Timestamps as strings | Converted to datetime | Required for time features |
| Negative time/distance | Set to NaN | Physically impossible |
| Extreme outliers | Winsorized at 99th pct | Preserve corridors, limit distortion |
| Hub name inconsistencies | Uppercase + strip | Prevent graph node splits |
| Null core columns | Rows dropped | Cannot use in graph or model |

### New columns created: 12
### Clean file saved to: `data/delivery_data_clean.csv`

---
### ➡️ Next: `03_graph_construction.ipynb`